# Debug Eye-Region Crop
Visualise `EyeRegionCropTransform` on a small batch of live and spoof (mask) images.

In [ ]:
import sys
sys.path.insert(0, '.')

import glob
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from miyagi_trainer.augmentations import EyeRegionCropTransform

In [ ]:
DATASET_ROOT = "/home/gfuhr/projects/data/liveness/controlid_liveness_ds_7_11b80/RGB"

live_files  = glob.glob(f"{DATASET_ROOT}/train/live/*.png")[:50]
spoof_files = glob.glob(f"{DATASET_ROOT}/train/spoof/controlid_spoof_masks_1_raw_*.png")[:50]
spoof_files += glob.glob(f"{DATASET_ROOT}/train/spoof/controlid_spoof_hard_mask_*.png")[:25]

random.seed(42)
samples = [(p, 'live')  for p in random.sample(live_files,  min(4, len(live_files)))] + \
          [(p, 'spoof') for p in random.sample(spoof_files, min(4, len(spoof_files)))]
print(f"Showing {len(samples)} images")

In [ ]:
import numpy as np
from torchvision import transforms as T

loose_crop = EyeRegionCropTransform(tight=False)
tight_crop = EyeRegionCropTransform(tight=True)
RESIZE_SIZES = [96, 112, 152, 172]

# ── helper: apply a resize_policy + size and return a PIL image ──────────────
from miyagi_trainer.augmentations import (
    eye_region_crop_exact, eye_region_crop_letterbox,
    eye_region_crop_tight_exact, eye_region_crop_tight_letterbox,
)

def apply_policy(policy_fn, size, img):
    """Run just the resize portion of a policy (no ToTensor/Normalize)."""
    pipe = policy_fn(size)
    # keep only the transforms up to (but not including) ToTensor
    resize_transforms = [t for t in pipe.transforms
                         if not isinstance(t, (T.ToTensor, T.Normalize))]
    for t in resize_transforms:
        img = t(img)
    return img

# ── main grid: rows = samples, cols = policies × sizes ───────────────────────
policies = [
    ("loose_exact",    eye_region_crop_exact),
    ("loose_letterbox",eye_region_crop_letterbox),
    ("tight_exact",    eye_region_crop_tight_exact),
    ("tight_letterbox",eye_region_crop_tight_letterbox),
]
n_cols = len(policies) * len(RESIZE_SIZES) + 1   # +1 for original

fig, axes = plt.subplots(len(samples), n_cols,
                         figsize=(3 * n_cols, 3 * len(samples)))

for row, (path, label) in enumerate(samples):
    img = Image.open(path).convert('RGB')
    w, h = img.size

    # col 0 – original with crop boxes
    ax = axes[row, 0]
    ax.imshow(img)
    for c, color, name in [(loose_crop, 'red', 'loose'), (tight_crop, 'lime', 'tight')]:
        xl  = round(w * c._x_margin_frac)
        xr  = w - xl
        yt  = max(0, c.EYE_Y_CENTER - c._y_margin)
        yb  = min(h, c.EYE_Y_CENTER + c._y_margin)
        ax.add_patch(patches.Rectangle((xl, yt), xr - xl, yb - yt,
                     linewidth=2, edgecolor=color, facecolor='none', label=name))
    ax.legend(fontsize=6, loc='lower right')
    ax.set_title(f"{label}\n{w}×{h}", fontsize=8)
    ax.axis('off')

    # remaining cols – one per (policy, size) combination
    col = 1
    for pol_name, pol_fn in policies:
        for sz in RESIZE_SIZES:
            out = apply_policy(pol_fn, sz, img.copy())
            ax = axes[row, col]
            ax.imshow(out)
            ax.set_title(f"{pol_name}\n→ {sz}×{sz}", fontsize=7)
            ax.axis('off')
            col += 1

plt.tight_layout()
plt.savefig('debug_eye_crop.png', dpi=80)
plt.show()
print("Saved debug_eye_crop.png")

In [ ]:
# Also show the ArcFace landmark positions overlaid on a live image
import numpy as np

# ArcFace reference in 272x272 (112 base + 80px border, scale ratio = 1.0)
arcface_pts_272 = np.array([
    [38.2946 + 80, 51.6963 + 80],   # left eye
    [73.5318 + 80, 51.5014 + 80],   # right eye
    [56.0252 + 80, 71.7366 + 80],   # nose
    [41.5493 + 80, 92.3655 + 80],   # left mouth
    [70.7299 + 80, 92.2041 + 80],   # right mouth
], dtype=np.float32)

img = Image.open(live_files[0]).convert('RGB')
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(img)

labels_pts = ['L-eye', 'R-eye', 'Nose', 'L-mouth', 'R-mouth']
colors_pts = ['cyan', 'cyan', 'yellow', 'magenta', 'magenta']
for (x, y), lbl, col in zip(arcface_pts_272, labels_pts, colors_pts):
    ax.plot(x, y, 'o', color=col, markersize=8)
    ax.text(x + 3, y, lbl, color=col, fontsize=7)

rect = patches.Rectangle(
    (0, crop.EYE_Y_CENTER - crop.EYE_Y_MARGIN),
    img.width, 2 * crop.EYE_Y_MARGIN,
    linewidth=2, edgecolor='red', facecolor='none'
)
ax.add_patch(rect)
ax.set_title("ArcFace landmarks + crop box", fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig('debug_eye_landmarks.png', dpi=100)
plt.show()
print("eye Y center:", crop.EYE_Y_CENTER, "  margin:", crop.EYE_Y_MARGIN)
print("crop box y:", crop.EYE_Y_CENTER - crop.EYE_Y_MARGIN, "->", crop.EYE_Y_CENTER + crop.EYE_Y_MARGIN)